# Download Planet Basemap Scenes for ARTS Using Orders API
This script downloads 2024 Planet Basemap grids to cover all ARTS v.6.0.0 polygons.

In [1]:
import os
import json
from google.cloud import storage
import google.auth
import gcsfs
import requests
import urllib.request
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely as shp
from pprint import pprint
import math
import time
import ast
import re
import sys
from datetime import datetime

In [2]:
# Get Planet API Key
%load_ext dotenv
%dotenv

api_key = os.getenv('PL_BM_API_KEY')
gcs_key = os.getenv("GCS_PL_ORDERS_KEY")

In [3]:
# setup session
session = requests.Session()

# authenticate
session.auth = (api_key, "")


In [16]:
# ! gcloud auth login
gcloud_creds, _ = google.auth.default()

In [17]:
storage_client = storage.Client(project="AbruptThawMapping")
bucket_name = "abrupt_thaw"
bucket = storage_client.bucket(bucket_name)

# Import Data

In [6]:
grids_filtered = gpd.read_file('../data/planet_grids_2024_artsv.6.0.0.geojson')
grids_filtered

,year,id,grid_column,grid_row,basemap_name,delivery_location,link,geometry
0,2024,0-1515,0,1515,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/0/1515/,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1950841.755 1938908.115, -1944872.9..."
1,2024,0-1516,0,1516,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/0/1516/,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1944872.974 1932975.846, -1938922.4..."
2,2024,0-1549,0,1549,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/0/1549/,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1757802.562 1747049.776, -1752423.2..."
3,2024,0-1553,0,1553,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/0/1553/,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1736383.836 1725762.072, -1731069.9..."
4,2024,1-1532,1,1532,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/1/1532/,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1857451.331 1834795.702, -1851767.6..."
...,...,...,...,...,...,...,...,...
23685,2024,2047-1516,2047,1516,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/2047/...,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1938933.535 1938933.535, -1933001.1..."
23686,2024,2047-1533,2047,1533,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/2047/...,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1840509.164 1840509.164, -1834877.3..."
23687,2024,2047-1536,2047,1536,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/2047/...,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1823665.148 1823665.148, -1818084.7..."
23688,2024,2047-1546,2047,1546,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/2047/...,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1768621.76 1768621.76, -1763209.464..."


In [7]:
imagery_year = '2024'

In [ ]:
start = datetime.now()
print("Downloads started at ", start)
delivery_directory = (
    "gs://"
    + bucket_name
    + "/planet_basemaps/global_quarterly/"
    + imagery_year
    + "/q3"
)

for index, row in grids_filtered.iterrows():
    basemap_name = row["basemap_name"]
    item_id = [row["id"]]  # only deliver a single item at a time
    delivery_location = row[
        "delivery_location"
    ]  # delivery_location includes the row number
    dir_path = (
        delivery_location.rsplit("/", 2)[0] + "/"
    )  # dir_path does not include the row number
    order_name = str(row["id"])  # order_name is the grid ID
    order_row_number = str(row["grid_row"]) # grid row needed to rename files later

    # because Planet packages orders in a folder with the order name,
    # I am using the row number as the order name, even though it will have duplicates
    # item_id will be saved in the order_log file to check for prior downloads

    # Get prior orders from GCS
    blobs = bucket.list_blobs(prefix = delivery_directory)
    files = [file for file in blobs if re.search("quad.tif", file.name)]
    prior_ids = [
        re.search("\\d{1,4}-\\d{1,4}", file.name).group() for file in files
    ]

    print("-------------------------------------")
    print(index, ": ", item_id[0])

    if item_id not in prior_ids:  # check if the basemap has already been delivered
        # create the order info
        order_info = {
            "name": order_name,
            "source_type": "basemaps",
            "products": [{"mosaic_name": basemap_name, "quad_ids": item_id}],
            "delivery": {
                "google_cloud_storage": {
                    "bucket": bucket_name,
                    "credentials": gcs_key,
                    "path_prefix": dir_path,
                }
            }
        }
        
        # Place order
        print(
            "Placing order for delivery to ", dir_path
        )
        request = requests.post('https://api.planet.com/compute/ops/orders/v2',
                                auth=(api_key, ''),
                                json=order_info)
        print(str(request))
        if str(request) in ['<Response [400]>', '<Response [401]>']:
            sys.exit("Invalid request or credentials. Check your request.")
        if str(request) == '<Response [409]>':
            while str(request) == '<Response [409]>':
                print("Concurrency error. Retrying in 30 seconds...")
                time.sleep(30)
                request = requests.post('https://api.planet.com/compute/ops/orders/v2',
                                        auth=(api_key, ''),
                                        json=order_info)
        if str(request) == '<Response [500]>': # server error, try again
            while str(request) == "<Response [500]>":
                print("Server error. Retrying in 30 seconds...")
                time.sleep(30)
                request = requests.post(
                    "https://api.planet.com/compute/ops/orders/v2",
                    auth=(api_key, ""),
                    json=order_info,
                )
        if str(request) == '<Response [202]>':
            print('Order has been placed.')
    else:
        print("Item already delivered. Skipping order.")
        
end = datetime.now()
print("Downloads finished at ", end)
total_time = end - start
print("Dowloading ", len(grids_filtered), " grids took ", total_time)


Downloads started at  2025-11-13 11:19:14.657232
-------------------------------------
0 :  0-1515
Placing order for delivery to  planet_basemaps/global_quarterly/2024/q3/0/
<Response [202]>
Order has been placed.
-------------------------------------
1 :  0-1516
Placing order for delivery to  planet_basemaps/global_quarterly/2024/q3/0/
<Response [202]>
Order has been placed.
-------------------------------------
2 :  0-1549
Placing order for delivery to  planet_basemaps/global_quarterly/2024/q3/0/
<Response [202]>
Order has been placed.
-------------------------------------
3 :  0-1553
Placing order for delivery to  planet_basemaps/global_quarterly/2024/q3/0/
<Response [202]>
Order has been placed.
-------------------------------------
4 :  1-1532
Placing order for delivery to  planet_basemaps/global_quarterly/2024/q3/1/
<Response [202]>
Order has been placed.
-------------------------------------
5 :  2-1532
Placing order for delivery to  planet_basemaps/global_quarterly/2024/q3/2/
<

In [2]:
# Dowloading  23690  grids took  10:07:55.428772.
elapsed_min = 10*60 + 7 + 55.43/60
grids_per_min = 23690/elapsed_min
grids_per_min

38.96869755887731

In [19]:
# rename files to desired name
blobs = [
    blob
    for blob in bucket.list_blobs(prefix="planet_basemaps/global_quarterly/2024/q3")
]
print(blobs[0])
# get correspondence between order IDs and Planet Quad row numbers
tif_files = [file.name for file in blobs if re.search("quad.tif", file.name)]
files = [file.name for file in blobs]
order_ids = [
    re.search("[a-z0-9]{8}(-[a-z0-9]{4}){3}-[a-z0-9]{12}", file)[0]
    for file in tif_files
]
order_row_numbers = [re.search("(?<=-)\\d{1,4}(?=_)", file)[0] for file in tif_files]
id_row_correspondence = dict(zip(order_ids, order_row_numbers))
pprint(id_row_correspondence)

# rename
pattern = re.compile("|".join(id_row_correspondence.keys()))
new_names = [
    re.sub(
        "global_quarterly_2024q3_mosaic/",
        "global_quarterly_2024q3_mosaic_",
        re.sub(pattern, lambda m: id_row_correspondence[m.group(0)], file),
    )
    for file in files
]
[bucket.rename_blob(blob, new_name) for blob, new_name in zip(blobs, new_names)]


<Blob: abrupt_thaw, planet_basemaps/global_quarterly/2024/q3/0/0c048c53-4aa2-446a-a665-85a530417088/global_quarterly_2024q3_mosaic/0-1516_metadata.json, 1763061531717262>
{'00029d20-b010-4255-a799-cd0231179675': '1656',
 '000793ef-0dfc-4092-8801-8e935c357acd': '1622',
 '000a5ea2-1301-429c-b136-ab2431f4e869': '1469',
 '000d218f-d6ec-40c4-bb66-5aef6f430d7f': '1349',
 '000de7a2-51db-45f0-91d9-5071f836920f': '1550',
 '00100dbc-7a58-4eb2-a33e-690ade410683': '1512',
 '0010ff47-650e-4d5a-83cd-15f89310f3d2': '1480',
 '00113c48-5b46-4d6f-ae4e-eb8f11bebef1': '1597',
 '0012db02-f8a0-4882-b0e7-d32b2c82227b': '1476',
 '00130c0a-a5eb-40d7-82ea-8117e02b94e5': '1333',
 '00159222-ac21-4a04-9127-37e9172a5261': '1632',
 '001895f0-fc69-43ab-91dc-613b76f996dd': '1596',
 '001c6789-b0b6-40ff-958b-2b4e7057944f': '1455',
 '0028506c-1177-424d-aaaf-7326b776dc94': '1379',
 '002ec0d0-6984-48e6-851e-6048a6d56fcc': '1393',
 '002fce1f-9db8-44a3-a3ef-9fda4e669467': '1357',
 '0030d7a7-05d3-4df5-93ab-1692782d9d2f': '157

[<Blob: abrupt_thaw, planet_basemaps/global_quarterly/2024/q3/0/1516/global_quarterly_2024q3_mosaic_0-1516_metadata.json, 1763139463284172>,
 <Blob: abrupt_thaw, planet_basemaps/global_quarterly/2024/q3/0/1516/global_quarterly_2024q3_mosaic_0-1516_ortho_udm2.tif, 1763139463792614>,
 <Blob: abrupt_thaw, planet_basemaps/global_quarterly/2024/q3/0/1516/global_quarterly_2024q3_mosaic_0-1516_provenance_raster.tif, 1763139464226381>,
 <Blob: abrupt_thaw, planet_basemaps/global_quarterly/2024/q3/0/1516/global_quarterly_2024q3_mosaic_0-1516_provenance_vector.zip, 1763139464707866>,
 <Blob: abrupt_thaw, planet_basemaps/global_quarterly/2024/q3/0/1516/global_quarterly_2024q3_mosaic_0-1516_quad.tif, 1763139465119967>,
 <Blob: abrupt_thaw, planet_basemaps/global_quarterly/2024/q3/0/1516/manifest.json, 1763139465631957>,
 <Blob: abrupt_thaw, planet_basemaps/global_quarterly/2024/q3/0/1515/global_quarterly_2024q3_mosaic_0-1515_metadata.json, 1763139466139851>,
 <Blob: abrupt_thaw, planet_basemaps/gl

In [ ]:
# # In case I need to cancel orders quickly
# requests.post('https://api.planet.com/compute/ops/bulk/orders/v2/cancel',
#               auth=(api_key, ''))

<Response [400]>